# Policy setup functions

This notebook has the code for every step of the setup and cleanup notebooks, so each of them is just a list of
steps:

| Notebook | Calls |
| --- | --- |
| `setup_policy_existing_workspaces` | `find_existing_workspaces` > `create_policy_set` > `add_rules` > `turn_on_policy_set` |
| `setup_policy_new_workspaces` | `check_capacity` > `create_workspaces` > `create_policy_set` > `add_rules` > `turn_on_policy_set` |
| `cleanup_policy` | `turn_off_policy_set` > `delete_workspaces` |

* **You don't run this notebook yourself.** Import it into the **same workspace** as the other notebooks, and keep
  its name `policy_setup_functions`. They load it with `%run policy_setup_functions`.
* To change **which items are allowed where**, edit the **Rules** section. Both setup notebooks use it.
* The code cells are collapsed. Select a cell to expand it.

## Helpers (used by every step)

* `call()` calls a Fabric REST API **as you**, the person running the notebook. No passwords or keys are needed.
  When creating the policy set and adding its rules, it also prints the URL and the exact JSON body it sends, so
  you can see it or copy it to reuse.
* `get_all()` reads every page of a list, for example all your workspaces.
* `stop()` prints what went wrong and what to do, and then stops the notebook.
* `find_capacity()` returns a capacity you have a role on, including your role.
* `stop_if_viewer_on_policy_store()` checks that you're a **Contributor** or higher on the **policy-store**
  workspace, where the policy set is created. A **Viewer** can't create items.

In [ ]:
import json
import time
import requests
import notebookutils   # built into every Fabric notebook

FABRIC_API = "https://api.fabric.microsoft.com/v1"
POWER_BI_API = "https://api.powerbi.com/v1.0/myorg"


class FabricError(Exception):
    """An error returned by a Fabric API, with its HTTP status and error code."""
    def __init__(self, status, code, message):
        self.status, self.code = status, code
        super().__init__(f"{message} (HTTP {status} {code})")


def call(method, url, body=None, show_json=False):
    """Calls a Fabric REST API as you, and returns the result. show_json=True prints the exact JSON sent."""
    if url.startswith("/"):
        url = FABRIC_API + url
    headers = {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}
    if show_json and body is not None:   # so it can be seen or copied to reuse
        print(f"   📄 {method} {url}\n" + "\n".join("      " + line for line in json.dumps(body, indent=2).splitlines()))
    response = requests.request(method, url, json=body, headers=headers)
    if response.status_code == 202:   # Fabric is still working on it: check every 3 seconds until it's done
        while True:
            time.sleep(3)
            operation = requests.get(response.headers["Location"], headers=headers).json()
            if operation.get("status") == "Succeeded":
                return {}
            if operation.get("status") in ("Failed", "Cancelled"):
                error = operation.get("error") or {}
                raise FabricError(500, error.get("errorCode", ""), error.get("message", "The operation failed."))
    if not response.ok:
        error = response.json() if "json" in response.headers.get("Content-Type", "") else {}
        error = error.get("error", error)   # Power BI APIs put the details under "error"
        raise FabricError(response.status_code, error.get("errorCode") or error.get("code", ""),
                          error.get("message") or response.text[:300])
    return response.json() if response.content else {}


def get_all(url):
    """Calls a Fabric list API and returns every item on every page."""
    items = []
    while url:
        page = call("GET", url)
        items += page.get("value", [])
        url = page.get("continuationUri")
    return items


class Stopped(Exception):
    pass


def stop(problem, what_to_do):
    """Prints what went wrong and how to fix it, and then stops the notebook."""
    print(f"\n❌ STOPPED: {problem}\n\n👉 What to do: {what_to_do}")
    raise Stopped(problem) from None


def get_my_capacities():
    """The capacities you have a role on. capacityUserAccessRight is your role: Admin, Assign (= Contributor) or None."""
    return call("GET", f"{POWER_BI_API}/capacities").get("value", [])


def find_capacity(capacity_id):
    """The capacity with this ID, or {} if you don't have a role on it."""
    for capacity in get_my_capacities():
        if capacity["id"].lower() == capacity_id.lower():
            return capacity
    return {}


def can_create_items_in(workspace):
    """True if you're a Contributor, Member or Admin of the workspace (a Viewer can't create items)."""
    mine = get_all("/workspaces?roles=Admin,Member,Contributor")
    return any(w["id"].lower() == workspace["id"].lower() for w in mine)


def stop_if_viewer_on_policy_store(policy_store_ws):
    """The policy set is created in the policy-store workspace, so you need Contributor or higher there."""
    name = policy_store_ws["displayName"]
    if not can_create_items_in(policy_store_ws):
        stop(f"You can't create the policy set in '{name}': your role there is Viewer.",
             f"Ask an Admin of workspace '{name}' to give you the Contributor, Member or Admin role.\n"
             "   (For the Data Engineering and Data Engineering + AI workspaces, the Viewer role is enough -\n"
             "   the notebook only reads them.)")
    print(f"   ✅ You can create the policy set in '{name}' (Contributor or higher).")

## Rules - change them here (used by both setup notebooks)

The rules are **allow** rules. Conditions inside one rule must **all** be true; an item is allowed if **any** rule
allows it. Item types use their Fabric API names, such as `DataPipeline` for a Data pipeline. Rule names can be at
most 60 characters long.

In [ ]:
def rule(name, *conditions):
    return {"displayName": name, "conditions": list(conditions), "effects": [{"type": "Allow"}]}


def condition(what, operator, values):
    return {"type": "Dynamic", "targetProperty": what, "predicate": {"operator": operator, "values": values}}


def build_rules(data_eng_ws, data_eng_ai_ws):
    return [
        rule("Allow Data Engineering items in both Data Eng workspaces",
             condition("workspace.id", "AnyOf", [data_eng_ws["id"], data_eng_ai_ws["id"]]),
             condition("item.type", "AnyOf", ["Lakehouse", "DataPipeline", "Notebook"])),
        rule("Allow Data agent in the Data Engineering + AI workspace",
             condition("workspace.id", "AnyOf", [data_eng_ai_ws["id"]]),
             condition("item.type", "AnyOf", ["DataAgent"])),
        rule("Allow all items in all other workspaces",
             condition("workspace.id", "NoneOf", [data_eng_ws["id"], data_eng_ai_ws["id"]])),
    ]

## Find existing workspaces (used by `setup_policy_existing_workspaces`)

`find_existing_workspaces()` finds the three workspaces by name, checks that you're a **Contributor** or higher on
the **policy-store** workspace (a **Viewer** role is enough on the **Data Engineering** and **Data Engineering +
AI** workspaces), checks that the Data Engineering and Data Engineering + AI workspaces are on the same Fabric
capacity and that it's running, and tells you whether you're a Capacity Admin.

In [ ]:
def find_existing_workspaces(policy_store_name, data_eng_name, data_eng_ai_name):
    """Finds the three workspaces and the capacity of the Data Engineering and Data Engineering + AI workspaces.
    Returns policy_store_ws, data_eng_ws, data_eng_ai_ws, capacity."""
    print("Finding your workspaces and their capacity...")
    all_workspaces = get_all("/workspaces")

    def find(name, role_needed):
        for workspace in all_workspaces:
            if workspace["displayName"].lower() == name.strip().lower():
                print(f"   ✅ Found '{workspace['displayName']}'")
                return workspace
        stop(f"Workspace '{name}' wasn't found, or you don't have access to it.",
             "Check the spelling in the Settings cell. If the name is right, ask the workspace Admin to give you\n"
             f"   the {role_needed} role on it.")

    policy_store_ws = find(policy_store_name, "Contributor (or Member or Admin)")
    data_eng_ws = find(data_eng_name, "Viewer (or higher)")
    data_eng_ai_ws = find(data_eng_ai_name, "Viewer (or higher)")
    stop_if_viewer_on_policy_store(policy_store_ws)
    capacity_id = data_eng_ws.get("capacityId") or ""
    if not capacity_id or capacity_id.lower() != (data_eng_ai_ws.get("capacityId") or "").lower():
        stop(f"'{data_eng_name}' and '{data_eng_ai_name}' aren't on the same Fabric capacity.",
             "Open each workspace > Workspace settings > License info, and assign both to the same Fabric capacity.")
    if not policy_store_ws.get("capacityId"):
        stop(f"'{policy_store_name}' isn't on a Fabric capacity, so it can't store a policy set.",
             "Open the workspace > Workspace settings > License info, and assign it to a Fabric capacity.")

    capacity = find_capacity(capacity_id) or {"id": capacity_id, "displayName": capacity_id}
    if capacity.get("state", "Active") != "Active":
        stop(f"Capacity '{capacity['displayName']}' is paused.",
             "Ask your Capacity Admin to resume it, then run the notebook again.")
    print(f"   ✅ Capacity: '{capacity['displayName']}' (ID {capacity_id})")
    if capacity.get("capacityUserAccessRight") == "Admin":
        print("   ✅ You're a Capacity Admin, so you can turn the policy set on.")
    else:
        print("   ⚠️ You're not a Capacity Admin. The policy set and rules are still created, and the last step\n"
              "      tells you what to send your Capacity Admin.")
    return policy_store_ws, data_eng_ws, data_eng_ai_ws, capacity

## Check the capacity and create workspaces (used by `setup_policy_new_workspaces`)

* `check_capacity()` checks that the capacity exists, is running, and that you can create workspaces on it
  (**Capacity Contributor** or **Capacity Admin**). If the ID is wrong, it lists the capacities you can use.
* `create_workspaces()` creates the three workspaces on the capacity; you're the Admin of the ones it creates.
  Workspaces that already exist are reused and **never moved**. Before creating anything, it stops if the **Data
  Engineering** or **Data Engineering + AI** workspace already exists on another capacity, or if the
  **policy-store** workspace already exists and you're only a **Viewer** there.

In [ ]:
def check_capacity(capacity_id):
    """Checks that you can create workspaces on the capacity. Returns the capacity."""
    print("Checking the capacity...")
    capacity_id = capacity_id.strip()
    capacity = find_capacity(capacity_id)
    if not capacity:
        usable = [f"      {c['displayName']}: {c['id']}" + ("" if c.get("state") == "Active" else "  (paused)")
                  for c in get_my_capacities() if c.get("capacityUserAccessRight") in ("Admin", "Assign")]
        stop(f"Capacity '{capacity_id}' wasn't found, or you don't have a role on it.",
             "Copy the right ID into CAPACITY_ID in the Settings cell. Capacities you can create workspaces on:\n"
             + ("\n".join(usable) or "      (none - ask your Capacity Admin to add you as a Capacity Contributor)"))
    if capacity.get("state") != "Active":
        stop(f"Capacity '{capacity['displayName']}' is paused.",
             "Ask your Capacity Admin to resume it, then run the notebook again.")
    print(f"   ✅ Capacity: '{capacity['displayName']}'")

    my_role = capacity.get("capacityUserAccessRight")   # Admin, Assign (= Contributor) or None
    if my_role not in ("Admin", "Assign"):
        stop(f"You can't create workspaces on capacity '{capacity['displayName']}'.",
             "Ask your Capacity Admin to either add you as a Capacity Contributor, or create the three workspaces for\n"
             "   you on this capacity and add you as their Admin. Then run this notebook again.")
    if my_role == "Admin":
        print("   ✅ You're a Capacity Admin, so you can create workspaces and turn the policy set on.")
    else:
        print("   ✅ You're a Capacity Contributor, so you can create workspaces.")
        print("   ⚠️ You're not a Capacity Admin. The policy set and rules are still created, and the last step\n"
              "      tells you what to send your Capacity Admin.")
    return capacity


def create_workspaces(capacity, policy_store_name, data_eng_name, data_eng_ai_name):
    """Creates the three workspaces on the capacity, reusing any that already exist.
    Returns policy_store_ws, data_eng_ws, data_eng_ai_ws."""
    print("Creating the workspaces...")
    existing = {workspace["displayName"].lower(): workspace for workspace in get_all("/workspaces")}

    # Check the workspaces that already exist first, so nothing is created if one can't be used.
    for name in (data_eng_name, data_eng_ai_name):   # both must be on this capacity
        workspace = existing.get(name.strip().lower())
        if workspace and (workspace.get("capacityId") or "").lower() != capacity["id"].lower():
            stop(f"Workspace '{name}' already exists, but on a different capacity.",
                 "Use a different workspace name in the Settings cell, or move the workspace to this capacity\n"
                 "   (Workspace settings > License info). The Data Engineering and Data Engineering + AI workspaces\n"
                 "   must be on the capacity the policy is for.")
    if policy_store_name.strip().lower() in existing:   # the policy set is created in the policy-store workspace
        stop_if_viewer_on_policy_store(existing[policy_store_name.strip().lower()])

    def get_or_create(name):
        if name.strip().lower() in existing:
            print(f"   ✅ '{name}' already exists, so it's reused.")
            return existing[name.strip().lower()]
        try:
            workspace = call("POST", "/workspaces", {"displayName": name.strip(), "capacityId": capacity["id"]})
        except FabricError as error:
            if "AlreadyExists" in error.code:
                stop(f"A workspace named '{name}' already exists, but you don't have access to it.",
                     "Use a different workspace name in the Settings cell, or ask that workspace's Admin to add you.")
            stop(f"Fabric didn't let you create workspace '{name}'.\n   Details: {error}",
                 "Creating a workspace needs BOTH of these:\n"
                 "   1. Permission to create workspaces - ask your Fabric administrator to turn on the\n"
                 "      'Create workspaces' tenant setting for you (Admin portal > Tenant settings).\n"
                 "   2. The Capacity Contributor or Capacity Admin role on the capacity (checked in the previous step).\n"
                 f"   Or ask your Capacity Admin to create '{name}' on capacity '{capacity['displayName']}' and add you\n"
                 "   as its Admin, then run this notebook again.")
        print(f"   ✅ Created '{name}'.")
        return workspace

    return get_or_create(policy_store_name), get_or_create(data_eng_name), get_or_create(data_eng_ai_name)

## Policy set steps (used by both setup notebooks)

* `create_policy_set()` creates the policy set in the **policy-store** workspace, or reuses it if it already
  exists. It's **off** until it's turned on.
* `add_rules()` sets the rules from the **Rules** section. Running it again replaces them, so they're never
  duplicated.
* `turn_on_policy_set()` activates it for the capacity. That needs the **Capacity Admin** role; without it, the
  policy set stays saved (but off), and a message for your Capacity Admin is printed. Only **one** policy set can be
  on per capacity.

In [ ]:
def create_policy_set(policy_store_ws, name):
    """Creates the policy set in the policy-store workspace, or reuses it if it already exists. Returns it."""
    store_name, policy_sets_url = policy_store_ws["displayName"], f"/workspaces/{policy_store_ws['id']}/policySets"
    print(f"Creating the policy set '{name}' in '{store_name}'...")

    def find():
        for item in get_all(policy_sets_url):
            if item["displayName"].lower() == name.lower():
                return item

    try:
        policy_set = find()
        if policy_set:
            print("   ✅ It already exists, so it's reused.")
            return policy_set
        call("POST", policy_sets_url, {
            "displayName": name,
            "description": "Controls which items can be created in the Data Engineering and "
                           "Data Engineering + AI workspaces.",
            "creationPayload": {"scope": {"type": "Capacity"}},
        }, show_json=True)
        print("   ✅ Created.")
        return find()
    except FabricError as error:
        stop(f"Fabric didn't let you create the policy set in '{store_name}'.\n   Details: {error}",
             "Creating a policy set needs BOTH of these:\n"
             f"   1. The Contributor, Member or Admin role on workspace '{store_name}' - ask the workspace Admin.\n"
             "   2. The 'Policies' tenant setting turned on for you - ask your Fabric administrator\n"
             "      (Admin portal > Tenant settings > Policies).\n"
             "   Also check that POLICY_SET_NAME has no spaces and isn't used by another item in the workspace.")


def add_rules(policy_store_ws, policy_set, data_eng_ws, data_eng_ai_ws):
    """Replaces the policy set's item creation rules with the rules from build_rules()."""
    print("Adding the rules...")
    rules = build_rules(data_eng_ws, data_eng_ai_ws)
    too_long = [r["displayName"] for r in rules if len(r["displayName"]) > 60]
    if too_long:
        stop("These rule names are longer than 60 characters, the most Fabric allows:\n"
             + "\n".join(f"      {name} ({len(name)} characters)" for name in too_long),
             "Shorten them in the Rules section, then run the notebook again.")
    rules_url = (f"/workspaces/{policy_store_ws['id']}/policySets/{policy_set['id']}"
                 "/policyRules/replaceByPolicy?beta=true")
    try:
        call("POST", rules_url, {"policy": "ItemCreation", "policyRules": rules}, show_json=True)
    except FabricError as error:
        stop(f"Fabric didn't accept the rules.\n   Details: {error}",
             "If the details mention a rule property, fix that rule in the Rules section. Otherwise, check that\n"
             "   you have the Contributor, Member or Admin role on workspace\n"
             f"   '{policy_store_ws['displayName']}'. Then run the notebook again.")
    for r in rules:
        print(f"   ✅ {r['displayName']}")


def turn_on_policy_set(policy_store_ws, policy_set, capacity):
    """Activates the policy set for the capacity, or explains what to send to a Capacity Admin."""
    capacity_name, store_name = capacity["displayName"], policy_store_ws["displayName"]
    print(f"Turning on the policy set for capacity '{capacity_name}'...")
    if policy_set.get("properties", {}).get("status") == "Active":
        print("   ✅ It's already on. Rule changes take effect within 15 minutes.")
        return
    try:
        call("POST", f"/workspaces/{policy_store_ws['id']}/policySets/{policy_set['id']}/activate",
             {"scopeType": "Capacity", "capacityId": capacity["id"]})
        print("   ✅ Turned on! It takes effect within 15 minutes.")
    except FabricError as error:
        if error.status == 409 or "Conflict" in error.code:
            stop(f"Another policy set is already turned on for capacity '{capacity_name}'.\n   Details: {error}",
                 "Only one policy set can be on per capacity. Your policy set and rules are saved (but off).\n"
                 "   Turn off the other policy set first (open it > Status > Inactive), then run the notebook again.")
        print(f"""
   ⚠️ Your policy set and rules are saved, but NOT turned on.
      Turning on a policy set needs the Capacity Admin role on capacity '{capacity_name}'.
      Details: {error}

   👉 Send this to your Capacity Admin:
      ------------------------------------------------------------------------------
      Please turn on the Fabric policy set '{policy_set['displayName']}' for capacity '{capacity_name}'
      (capacity ID {capacity['id']}). It's in workspace '{store_name}'.
      Steps: open workspace '{store_name}' > open '{policy_set['displayName']}' > Status > Active >
      choose capacity '{capacity_name}'. It takes effect within 15 minutes.
      ------------------------------------------------------------------------------""")

## Clean up (used by `cleanup_policy`)

* `turn_off_policy_set()` turns off (deactivates) the policy set stored in the **policy-store** workspace. That
  needs the **Capacity Admin** role. If it fails, the notebook stops, so no workspace is deleted while the policy
  is still on.
* `delete_workspaces()` deletes the three workspaces and everything in them, but **only** if you ask for it. That
  needs the **Admin** role on each workspace.

Workspaces or a policy set that don't exist are skipped, so it's safe to run again.

In [ ]:
def find_workspace(name):
    """The workspace with this name, or None if it doesn't exist or you can't see it."""
    for workspace in get_all("/workspaces"):
        if workspace["displayName"].lower() == name.strip().lower():
            return workspace


def turn_off_policy_set(policy_store_name, policy_set_name):
    """Turns off (deactivates) the policy set stored in the policy-store workspace."""
    print(f"Turning off the policy set '{policy_set_name}' in '{policy_store_name}'...")
    policy_store_ws = find_workspace(policy_store_name)
    if not policy_store_ws:
        print(f"   ✅ Workspace '{policy_store_name}' doesn't exist (or you can't see it), so there's nothing to\n"
              "      turn off.")
        return
    policy_sets_url = f"/workspaces/{policy_store_ws['id']}/policySets"
    try:
        policy_set = next((p for p in get_all(policy_sets_url)
                           if p["displayName"].lower() == policy_set_name.lower()), None)
        if not policy_set:
            print("   ✅ The policy set doesn't exist, so there's nothing to turn off.")
            return
        if policy_set.get("properties", {}).get("status") != "Active":
            print("   ✅ It's already off.")
            return
        call("POST", f"{policy_sets_url}/{policy_set['id']}/deactivate", {"scopeType": "Capacity"})
    except FabricError as error:
        if error.code == "PolicySetNotActiveOnScope":
            print("   ✅ It's already off.")
            return
        stop(f"Fabric didn't let you turn off the policy set in '{policy_store_name}'.\n   Details: {error}",
             "Turning off a policy set needs the Capacity Admin role on its capacity (and access to workspace\n"
             f"   '{policy_store_name}'). Ask your Capacity Admin to turn it off (open workspace\n"
             f"   '{policy_store_name}' > open '{policy_set_name}' > Status > Inactive), then run this notebook\n"
             "   again. Nothing was deleted.")
    print("   ✅ Turned off. Item creation is no longer restricted (within 15 minutes).")


def delete_workspaces(policy_store_name, data_eng_name, data_eng_ai_name, really_delete=False):
    """Deletes the three workspaces and everything in them - only if really_delete is True."""
    print("Deleting the workspaces...")
    if not really_delete:
        print("   ⏭️ Skipped, because DELETE_WORKSPACES = False. The workspaces and everything in them are kept.")
        return
    for name in (policy_store_name, data_eng_name, data_eng_ai_name):
        workspace = find_workspace(name)
        if not workspace:
            print(f"   ✅ '{name}' doesn't exist (or you can't see it), so it's skipped.")
            continue
        try:
            call("DELETE", f"/workspaces/{workspace['id']}")
        except FabricError as error:
            stop(f"Fabric didn't let you delete workspace '{name}'.\n   Details: {error}",
                 f"Deleting a workspace needs the Admin role on it. Ask an Admin of '{name}' to make you an Admin,\n"
                 "   or to delete it for you. Then run this notebook again - deleted workspaces are skipped.")
        print(f"   ✅ Deleted '{name}' and everything in it.")


print("✅ Policy setup functions loaded")